__Installing Pysaprk and importing libraries__

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Creating  a Spark session


In [3]:
spark = SparkSession.builder \
    .appName("CustomerDataProcessing") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 19:05:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
print(spark.version)

4.1.2


 ## Load a CSV file into a Spark DataFrame



In [5]:
file_path="../Data/Superstore.csv"
df = spark.read.csv(file_path,header=True,inferSchema=True,quote='"',escape='"',multiLine=True)
df.show(8)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [6]:
df.describe()


DataFrame[summary: string, Row ID: string, Order ID: string, Order Date: string, Ship Date: string, Ship Mode: string, Customer ID: string, Customer Name: string, Segment: string, Country: string, City: string, State: string, Postal Code: string, Region: string, Product ID: string, Category: string, Sub-Category: string, Product Name: string, Sales: string, Quantity: string, Discount: string, Profit: string]

In [7]:
df.count()

9994

In [8]:
print(df.columns)
df.printSchema()

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |--

## Transforming  Data 
__Renamed Columns__   
__Changed Data Type__ 

In [9]:
df = df.withColumnRenamed("Sales", "Price")

cols_to_cast = ["Price", "Quantity", "Discount", "Profit"]
for c in cols_to_cast:
    df = df.withColumn(c, col(c).cast("double"))

date_cols = ["Ship Date", "Order Date"]
for d in date_cols:
    df = df.withColumn(d, to_date(col(d), "M/d/yyyy"))



**Verifying Structure**

In [11]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



__Data Cleaning__

In [12]:
df=df.dropDuplicates()

df.na.drop()

df.na.fill("Unknown")

# remove invalid dates
df = df.filter(col("Ship Date").isNotNull())

df.show(15)

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Price|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|   281|US-2015-161991|2015-09-26|2015-09-28|  Second Class|   SC-20725|Steven Cartwright|   Consumer|United States|      Houston|         Texas|      77070|Central|OFF-BI-10004967|Office Supplies|  

In [13]:
df.count()

9994

__Analysing Data__

In [14]:
Furniture_df = df.filter(col("Category") == "Furniture")

Furniture_df.show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+---------------+-------------+-----------+-------+---------------+---------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|           City|        State|Postal Code| Region|     Product ID| Category|Sub-Category|        Product Name|  Price|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+---------------+-------------+-----------+-------+---------------+---------+------------+--------------------+-------+--------+--------+---------+
|  3039|US-2015-123960|2015-06-11|2015-06-16|Standard Class|   BD-11605|     Brian Dahlen|   Consumer|United States|         Monroe|    Louisiana|      71203|  South|FUR-FU-10004666|Furniture| Furnishings|DAX Cl

__Average earned by Store__

In [15]:
df.select( avg("Price")).show()

+----------------+
|      avg(Price)|
+----------------+
|229.858000830497|
+----------------+



__No of items sold in different Regions__ 

In [16]:
df.groupBy("Region").count().show()

+-------+-----+
| Region|count|
+-------+-----+
|  South| 1620|
|Central| 2323|
|   East| 2848|
|   West| 3203|
+-------+-----+



__Items Sold in Category in different Regions__

In [17]:
df.groupBy("Region","Category") \
  .sum("Price") \
  .show()

+-------+---------------+------------------+
| Region|       Category|        sum(Price)|
+-------+---------------+------------------+
|  South|Office Supplies|125651.31299999992|
|Central|Office Supplies|167026.41500000015|
|   West|Office Supplies|220853.24900000019|
|  South|      Furniture|117298.68399999992|
|Central|     Technology|170416.31199999992|
|Central|      Furniture|163797.16379999998|
|  South|     Technology|        148771.908|
|   West|     Technology| 251991.8320000001|
|   West|      Furniture|252612.74349999995|
|   East|Office Supplies|205516.05499999985|
|   East|      Furniture|208291.20400000003|
|   East|     Technology|        264973.981|
+-------+---------------+------------------+



## Loading (Saving data to another location)

In [19]:
df.write.mode("overwrite").csv("output")

In [20]:
import os

print(os.listdir())

['output', '.ipynb_checkpoints', 'Spark_Assignment5.ipynb']


## Summary

 1 end-to-end Spark pipeline on the Superstore dataset — loaded raw CSV data.  
 2 handled malformed text fields with quote/escape parsing, transformed column types.  
 3 cleaned duplicates and nulls, then ran filtering and aggregation queries before  
 4 Loaded (saving the processed output).

**Key Observation:**  
   1  The biggest challenge was the CSV parsing issue — product names containing
commas inside quotes shifted columns during ingestion, causing numeric casts to fail
on text values. Fixing this required explicit `quote` and `escape` parameters rather
than relying on `inferSchema` alone. This reinforced why data validation matters
before transformation in any production pipeline.

 2  Spark's in-memory processing and DataFrame API made aggregations like
`groupBy("Region","Category").sum("Price")` fast and concise compared to
row-by-row processing — the core advantage over MapReduce.